# LangGraph 노트북 멀티턴 챗봇 · 데모

폐쇄망 친화형 **멀티턴 챗봇 + 관찰성(observability) 트레이서 + HTML 반출** 을 `chatbot.py` 하나로 모아둔 변환물입니다. 바로 다음 셀 하나만 실행하면 **모든 기능을 위젯 안에서** 체험할 수 있습니다.

## ⚡ 빠른 시작 — 아래 두 단계면 끝

1. **다음 code 셀을 실행** → 셀 output 안에 입력창이 생깁니다.
2. **아래 치트시트의 예시 문구를 입력창에 그대로 복사** 해 넣고 **보내기** 를 누르세요.

## 🎯 기능 치트시트 — `chat_ui()` 에 그대로 입력해보세요

| # | 기능 | 입력 예시 | 어떻게 동작? |
|---|---|---|---|
| 1 | **일반 대화** | `안녕, 자기소개 부탁해` | MockLLM 응답이 채팅 풍선으로 쌓입니다 |
| 2 | **멀티턴 맥락 유지** | `방금 내가 뭐라고 했는지 요약해줘` | 같은 `thread_id` 로 이전 메시지 자동 참조 |
| 3 | **Tool 호출 (calculator)** | `12 + 7 + 100 계산해줘` | `tool:calculator` span 이 트레이스에 기록됨 |
| 4 | **HITL — 객관식 (RadioButtons)** | `포트폴리오 추천해줘` | 입력창 → 라디오 버튼 (**하나** 선택) |
| 5 | **HITL — 주관식 (Textarea)** | `상황을 구체적으로 설명해줘` | 입력창 → 자유 텍스트 입력 |
| 6 | **HITL — 복수선택 (Checkbox 그룹)** | `관심 있는 자산군 여러 개 알려줘` | 입력창 → 체크박스 (**여러 개** 선택 가능) |
| 7 | **트레이스 뷰어** | **[트레이스 보기]** 버튼 클릭 | chain / LLM / tool span 트리 + 토큰 / latency |
| 8 | **새 대화 (thread 리셋)** | **[새 대화]** 버튼 클릭 | 맥락 끊기 (Tracer 는 유지) |

### 💡 HITL 트리거 키워드 전체 목록

MockLLM 은 다음 키워드를 만나면 해당 유형으로 되묻습니다 (실제 사내 LLM 어댑터에서는 원하는 로직으로 대체).

- **복수선택**: `여러` · `복수` · `해당` · `체크` · `모두` — *먼저 검사됨*
- **객관식**: `추천` · `고를` · `고르` · `골라` · `선택지` · `옵션`
- **주관식**: `설명해` · `알려줘` · `명확` · `모호` · `구체적`

### 🔒 폐쇄망 친화 포인트

- 단일 파일 `chatbot.py` 만 반입하면 끝 (LICENSE 포함)
- 외부 네트워크 호출 / 바이너리 영속화 **없음** — 모든 기록은 self-contained HTML
- `save_trace("trace.html")` 로 만든 파일은 `<script src>` · `fetch()` · `<link href>` **일절 없음** → 업무망 `file://` 에서 바로 열림


## 1️⃣ 메인: `chat_ui()` — 이 셀 하나로 모든 기능 체험

> 위 치트시트의 예시를 입력창에 **복사/붙여넣기** 해서 하나씩 눌러보세요.
> 입력창이 상황에 따라 **Textarea / RadioButtons / Checkbox 그룹** 으로 자동 전환됩니다.

필요 패키지: `ipywidgets` (JupyterLab 에 기본 번들되는 경우가 많음)


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from chatbot import Chatbot

bot = Chatbot()     # MockLLM + 새 Tracer + 랜덤 thread_id
bot.chat_ui()       # ⬅️ 여기 입력창에 치트시트의 예시 문구를 넣어보세요


## 2️⃣ 체험 후: 트레이스를 HTML 한 장으로 반출

업무망 반입용 self-contained HTML 로 저장합니다. 생성된 `trace.html` 은 브라우저로 바로 열립니다 (외부 fetch 없음).


In [ ]:
path = bot.save_trace("trace.html")
print("저장됨:", path)

# 통계 요약
import json
print(json.dumps(bot.summary(), ensure_ascii=False, indent=2))


---

## 📚 아래는 **프로그래매틱 API** 참고 (선택 · 개발자용)

위의 `chat_ui()` 가 모든 기능을 커버하므로 **굳이 실행할 필요 없습니다.** 사내 LLM 어댑터를 작성하거나, 위젯 없이 자동화 파이프라인에 챗봇을 끼워넣을 때 참고용으로 두었습니다.


### A. 기본 대화 + 멀티턴 + tool 호출

`bot.chat(user_message)` 는 한 턴의 최종 assistant 응답 문자열을 돌려줍니다. 같은 `thread_id` 로 호출하면 `MemorySaver` 가 이전 메시지를 자동 복원합니다.


In [ ]:
api_bot = Chatbot()            # 위 bot 과 thread 분리
print("[턴1]", api_bot.chat("안녕하세요, 자기소개 부탁드립니다."))
print("[턴2]", api_bot.chat("12 + 7 + 100 을 계산해줘. 결과만 한 줄로."))   # tool 트리거
print("[턴3]", api_bot.chat("방금 내가 어떤 질문을 두 개 했는지 요약해줘."))  # 멀티턴 맥락


### B. 대화 이력과 트레이스를 셀 안에 시각화

`show_history()` · `show_trace()` 는 `IPython.display` 로 **self-contained HTML** 을 셀 output 에 바로 렌더합니다.


In [ ]:
api_bot.show_history()


In [ ]:
api_bot.show_trace()


### C. 새 thread 로 리셋 — 대화 맥락 끊기

`bot.reset()` 은 새 `thread_id` 를 발급하며, `MemorySaver` 는 thread 단위로 상태를 관리하므로 이전 대화와 완전히 분리됩니다. Tracer 는 유지되며 `clear_trace()` 로 비울 수 있습니다.


In [ ]:
new_id = api_bot.reset()
print("new thread_id:", new_id)
print(api_bot.chat("새 대화 시작! 이전 대화는 기억 안 나겠지?"))


### D. HITL — `chat_ui()` 없이 코드로 `resume()` 호출

사내 자동화 파이프라인에서는 위젯 없이 `chat()` → `pending_interrupt` 검사 → 자체 UI(또는 CLI/TUI) 로 사용자 응답 수집 → `resume(answer)` 패턴으로 HITL 을 다룹니다. 아래는 그 최소 형태입니다.

- `type == "choice"` → `answer` 는 `options` 중 **하나**
- `type == "multi_choice"` → `answer` 는 `options` 의 **부분집합 list**
- `type == "input"` → `answer` 는 **임의의 문자열**

> 💡 인터랙티브 체험은 맨 위 `bot.chat_ui()` 에서 하세요. 이 셀은 어디까지나 **API 사용법 참고용** 입니다.


In [ ]:
hbot = Chatbot()
first = hbot.chat("관심 있는 자산군 여러 개 알려줘")   # multi_choice 트리거
ask = hbot.pending_interrupt
print("LLM 질문:", first)
print("ask 페이로드:", ask)

# --- 실제 사내 코드에서는 여기서 자체 UI 를 띄워 사용자 응답을 받습니다. ---
# (이 노트북에서는 '앞에서 두 개를 체크한 척' 하는 시뮬레이션)
if ask["type"] == "multi_choice":
    answer = ask["options"][:2]
elif ask["type"] == "choice":
    answer = ask["options"][0]
else:
    answer = "임의의 자유 텍스트 응답"

print("사용자 응답:", answer)
print("최종 응답 :", hbot.resume(answer))


### E. 사내 실제 LLM 어댑터로 `MockLLM` 교체

아래 인터페이스를 따르는 클래스를 주입하면 `MockLLM` 을 갈아끼울 수 있습니다. 사내 LLM 호출 자체를 `tracer.span(...)` 으로 감싸주면 자동으로 트레이스 뷰어의 `LLM` span 으로 집계됩니다.


In [ ]:
from chatbot import Chatbot, Tracer


class MyLocalLLM:
    """사내 실제 LLM 을 감싸는 어댑터 템플릿.

    필요 메서드: invoke(messages: list[dict]) -> dict
      - messages: [{"role": "user"|"assistant"|"tool", "content": "..."}, ...]
      - 반환   : {"role": "assistant", "content": "..."}
                 HITL 이 필요하면 "ask_user" 키를 추가: {"type": "choice"|"multi_choice"|"input", "question": ..., "options": [...]}
    """

    def __init__(self, tracer: Tracer | None = None):
        self.tracer = tracer
        # TODO: 실제 사내 LLM 클라이언트 핸들 초기화 (URL / 인증 / 모델명 등)

    def invoke(self, messages: list[dict]) -> dict:
        # --- 실제 사내 LLM 호출 로직으로 교체하세요 ---
        out_text = "이 자리에 사내 LLM 응답을 넣으세요."
        tokens_in = sum(len(str(m.get("content", "")).split()) for m in messages)
        tokens_out = len(out_text.split())

        reply = {"role": "assistant", "content": out_text}
        # HITL 이 필요한 경우 (옵션):
        # reply["ask_user"] = {"type": "multi_choice", "question": "...", "options": [...]}

        # --- 트레이스 기록 (tracer 가 주입된 경우 자동으로 LLM span 으로 집계됨) ---
        if self.tracer is not None:
            with self.tracer.span("LLM:MyLocal", kind="llm",
                                  inputs=messages,
                                  metadata={"model": "my-local"}) as s:
                self.tracer.finish(s, outputs=reply,
                                   tokens_in=tokens_in, tokens_out=tokens_out)
        return reply


# 사용 예 (실제 돌리려면 invoke() 의 TODO 부분을 사내 LLM 호출로 교체)
# my_bot = Chatbot(llm=MyLocalLLM())
# my_bot.chat_ui()
print("어댑터 템플릿 정의 완료. invoke() 의 out_text 를 사내 LLM 호출로 교체하세요.")
